# Python LLM APIs

The four ways we call Large Language Models from Python in this course:

| Provider | Where it runs | Key library |
|---|---|---|
| **Ollama** | Local machine | `ollama` |
| **OpenAI** | Remote (OpenAI cloud) | `openai` |
| **OpenRouter** | Remote (gateway to 100+ models) | `openai` (compatible) |
| **LiteLLM** | Unified wrapper for all of the above | `litellm` |

Other providers work the same way, either through their own SDK or through LiteLLM:

| Provider | Docs |
|---|---|
| Anthropic Claude | https://docs.anthropic.com/en/api/getting-started |
| Hugging Face Inference | https://huggingface.co/docs/inference-providers |
| Google Gemini | https://ai.google.dev/gemini-api/docs |
| Mistral | https://docs.mistral.ai/api/ |

> API keys for cloud providers should be stored in a `.env` file or as environment variables and **never hard-coded**.


In [ ]:
%pip install -q ollama openai litellm python-dotenv pyyaml


In [ ]:
import os
from dotenv import load_dotenv

# Load API keys from a .env file (create one with your keys)
load_dotenv()

# Load central configuration (config.yaml, shared with the RAG notebooks)
import rag_common as rc
cfg = rc.load_config()

# Keys expected in .env:
#   OPENAI_API_KEY=sk-...
#   OPENROUTER_API_KEY=sk-or-v1-...


---
## 1. Ollama — Local Models

[Ollama](https://ollama.com) lets you run open-weight models (Llama 3, Phi-3, Mistral, Gemma, etc.) **entirely on your machine** — no API key, no data sent to the cloud.

### Setup
1. Install Ollama from https://ollama.com/download  
2. Pull a model: `ollama pull llama3.2` (or `phi3`, `mistral`, `gemma3`, …)  
3. Make sure the Ollama daemon is running (`ollama serve`)

The Python `ollama` library talks to the local daemon over HTTP.

In [ ]:
try:
    import ollama
except ImportError:
    ollama = None
    print('⚠️ Package "ollama" not installed. Run: pip install ollama')

# List all locally available models using config URL
ollama_url = cfg["ollama"]["url"]
ollama_client = ollama.Client(host=ollama_url) if ollama else None

if ollama_client:
    try:
        models_resp = ollama_client.list()
        print('Locally installed models:')
        models = getattr(models_resp, 'models', None)
        if models is None and isinstance(models_resp, dict):
            models = models_resp.get('models', [])
        if models:
            for m in models:
                name = getattr(m, 'model', None) or getattr(m, 'name', None) or (m.get('model') if isinstance(m, dict) else str(m))
                size = getattr(m, 'size', None) or (m.get('size', 0) if isinstance(m, dict) else 0)
                print(f'  {name}  ({size / 1e9:.1f} GB)')
        else:
            default_model = cfg["ollama"]["models"][0]
            print(f'  (No models installed yet. Pull one via: ollama pull {default_model})')
    except Exception as e:
        print(f'⚠️ Ollama server not reachable on {ollama_url} ({e})')
        print('  To run local models, start Ollama desktop or run `ollama serve` in a terminal.')
else:
    print('Skipping local model listing: ollama library not installed.')


In [ ]:
try:
    from ollama import chat, ChatResponse
except ImportError:
    chat, ChatResponse = None, None

OLLAMA_MODEL = cfg["ollama"]["models"][0]

def ollama_chat(prompt: str, model: str = OLLAMA_MODEL, system: str = '') -> str:
    """Single-turn call to a local Ollama model with graceful fallback."""
    if not ollama_client:
        return '[ollama package not installed. Run `pip install ollama` to test local models.]'
    try:
        messages = []
        if system:
            messages.append({'role': 'system', 'content': system})
        messages.append({'role': 'user', 'content': prompt})
        response = ollama_client.chat(model=model, messages=messages)
        return response.message.content
    except Exception as e:
        return f'[Ollama unavailable: {e}. Start Ollama and run `ollama pull {model}` to test.]'

print(ollama_chat('Explain what an autoregressive language model is in 2 sentences.'))


In [ ]:
# Ollama: System prompt + multi-turn conversation
messages = [
    {'role': 'system', 'content': 'You are a concise NLP tutor. Answer in bullet points only.'},
    {'role': 'user',   'content': 'What is the role of the temperature parameter in LLM sampling?'},
]

if ollama_client:
    try:
        response = ollama_client.chat(model=OLLAMA_MODEL, messages=messages)
        print('Turn 1 response:\n', response.message.content)
        messages.append({'role': 'assistant', 'content': response.message.content})
        messages.append({'role': 'user',      'content': 'What happens if temperature is set to exactly 0.0?'})
        response2 = ollama_client.chat(model=OLLAMA_MODEL, messages=messages)
        print('\nTurn 2 response:\n', response2.message.content)
    except Exception as e:
        print(f'[Ollama multi-turn conversation skipped: {e}]')
else:
    print('[Ollama multi-turn skipped: ollama package not installed]')


In [ ]:
# Ollama: Streaming response (tokens printed as they arrive)
print('Streaming response:\n')
if ollama_client:
    try:
        stream = ollama_client.chat(
            model=OLLAMA_MODEL,
            messages=[{'role': 'user', 'content': 'Write a haiku about pretraining scaling laws.'}],
            stream=True,
        )
        for chunk in stream:
            print(chunk.message.content, end='', flush=True)
        print()
    except Exception as e:
        print(f'\n[Ollama streaming skipped: {e}]')
else:
    print('[Ollama streaming skipped: ollama package not installed]')


---
## 2. OpenAI API

The `openai` Python SDK gives access to GPT-4o, GPT-4 Turbo, GPT-3.5-Turbo, and the Assistants/Embeddings/Vision APIs.

```
OPENAI_API_KEY=sk-...   ← set in .env
```

In [ ]:
try:
    from openai import OpenAI
except ImportError:
    OpenAI = None
    print('⚠️ Package "openai" not installed. Run: pip install openai')

openai_api_key = os.getenv('OPENAI_API_KEY')
openai_client = OpenAI(api_key=openai_api_key) if (OpenAI and openai_api_key) else None
OPENAI_MODEL = cfg["openai"]["model"]

def openai_chat(prompt: str, model: str = OPENAI_MODEL, system: str = 'You are a helpful assistant.') -> str:
    if not OpenAI:
        return '[openai package not installed. Run: pip install openai]'
    if not openai_client:
        return '[Skipping: OPENAI_API_KEY not configured in .env]'
    try:
        response = openai_client.chat.completions.create(
            model=model,
            messages=[
                {'role': 'system', 'content': system},
                {'role': 'user',   'content': prompt},
            ],
        )
        return response.choices[0].message.content
    except Exception as e:
        return f'[OpenAI Error: {e}]'

print(openai_chat('Explain what an attention head does in a transformer in 2 sentences.'))


In [ ]:
# OpenAI: Streaming chat
print('Streaming GPT response:\n')
if openai_client and openai_api_key:
    try:
        stream = openai_client.chat.completions.create(
            model=OPENAI_MODEL,
            messages=[{'role': 'user', 'content': 'Give 3 real-world use cases of Mixture-of-Experts (MoE) models.'}],
            stream=True,
        )
        for chunk in stream:
            delta = chunk.choices[0].delta.content
            if delta:
                print(delta, end='', flush=True)
        print()
    except Exception as e:
        print(f'[OpenAI Streaming Error: {e}]')
else:
    print('[Skipping streaming: OPENAI_API_KEY not configured]')


In [ ]:
# OpenAI Embeddings — useful for semantic search & RAG
texts = [
    'The cat sat on the mat.',
    'A feline rested on a rug.',
    'Pretraining language models requires petabytes of web text.',
]

OPENAI_EMBED_MODEL = cfg["openai"]["embedding_model"]

if openai_client and openai_api_key:
    try:
        embeddings_response = openai_client.embeddings.create(
            model=OPENAI_EMBED_MODEL,
            input=texts,
        )
        print(f'Embedded {len(texts)} texts.')
        print(f'Embedding dimension: {len(embeddings_response.data[0].embedding)}')
        print(f'First 5 values of text 0: {embeddings_response.data[0].embedding[:5]}')
    except Exception as e:
        print(f'[OpenAI Embeddings Error: {e}]')
else:
    print('[Skipping embeddings: OPENAI_API_KEY not configured]')


---
## 3. OpenRouter — Unified Remote Gateway

[OpenRouter](https://openrouter.ai) provides a **single OpenAI-compatible endpoint** that routes to 100+ models: GPT-4o, Claude 3.5, Llama, Gemini, Mistral, Qwen, DeepSeek, and more.

Key advantages:
- One API key, many providers
- Automatic fallbacks between providers
- Per-request model selection
- Competitive pricing (some models free)

```
OPENROUTER_API_KEY=sk-or-v1-...   ← set in .env
```

Base URL: `https://openrouter.ai/api/v1`

In [ ]:
OPENROUTER_BASE_URL = cfg["openrouter"]["base_url"]
OPENROUTER_MODEL = cfg["openrouter"]["model"]
openrouter_key = os.getenv('OPENROUTER_API_KEY')

or_client = OpenAI(
    base_url=OPENROUTER_BASE_URL,
    api_key=openrouter_key or 'placeholder',
) if OpenAI else None

def openrouter_chat(prompt: str,
                    model: str = OPENROUTER_MODEL,
                    system: str = 'You are a helpful assistant.') -> str:
    if not or_client:
        return '[openai package required to call OpenRouter. Run: pip install openai]'
    if not openrouter_key:
        return '[Skipping OpenRouter: OPENROUTER_API_KEY not configured in .env]'
    try:
        response = or_client.chat.completions.create(
            model=model,
            messages=[
                {'role': 'system', 'content': system},
                {'role': 'user',   'content': prompt},
            ],
        )
        return response.choices[0].message.content
    except Exception as e:
        return f'[OpenRouter Error: {e}]'

print(openrouter_chat('What is perplexity in language model evaluation?'))


In [ ]:
# OpenRouter: Compare the core open model families (Mistral, Gemma, Qwen, LLaMA)
PROMPT = 'In one sentence, define what an attention mechanism does in transformer pretraining.'

models_to_compare = cfg["openrouter"]["models_to_compare"]

if openrouter_key:
    for label, model_id in models_to_compare.items():
        try:
            answer = openrouter_chat(PROMPT, model=model_id)
            print(f'\n[{label}]\n{answer}')
        except Exception as e:
            print(f'\n[{label}] Error: {e}')
else:
    print('[Set OPENROUTER_API_KEY in .env to run live side-by-side comparison across Mistral, Gemma, Qwen, and LLaMA]')


In [ ]:
# OpenRouter: Streaming response
print('Streaming OpenRouter response:\n')
if openrouter_key:
    try:
        stream = or_client.chat.completions.create(
            model=OPENROUTER_MODEL,
            messages=[{'role': 'user', 'content': 'Write a short poem about tokenization.'}],
            stream=True,
        )
        for chunk in stream:
            delta = chunk.choices[0].delta.content
            if delta:
                print(delta, end='', flush=True)
        print()
    except Exception as e:
        print(f'[OpenRouter Streaming Error: {e}]')
else:
    print('[Skipping streaming: OPENROUTER_API_KEY not configured]')


---
## 4. LiteLLM — One Interface for All Providers

[LiteLLM](https://github.com/BerriAI/litellm) wraps all the providers above (and more) behind a single `completion()` call that mirrors the OpenAI API.  
This is handy when you want provider-agnostic code.

```python
litellm.completion(model="gpt-4o", ...)           # OpenAI
litellm.completion(model="claude-3-5-haiku...", ...)   # Anthropic
litellm.completion(model="ollama/llama3.2", ...)   # Ollama (local)
litellm.completion(model="openrouter/...", ...)    # OpenRouter
```

In [ ]:
try:
    import litellm
    litellm.set_verbose = False
except ImportError:
    litellm = None
    print('⚠️ Package "litellm" not installed. Run: pip install litellm')

def litellm_chat(prompt: str, model: str, system: str = 'You are a helpful assistant.') -> str:
    if not litellm:
        return '[litellm package not installed. Run: pip install litellm]'
    try:
        response = litellm.completion(
            model=model,
            messages=[
                {'role': 'system', 'content': system},
                {'role': 'user',   'content': prompt},
            ],
        )
        return response.choices[0].message.content
    except Exception as e:
        return f'[LiteLLM Error ({model}): {e}]'

QUESTION = 'What is the intuition behind TF-IDF?'

litellm_models = cfg["litellm"]["models"]

for model_str in litellm_models:
    answer = litellm_chat(QUESTION, model=model_str)
    print(f'\n[{model_str}]\n{answer}')


---
## Summary & Choosing the Right API

| Criterion | Ollama | OpenAI | OpenRouter | LiteLLM |
|---|---|---|---|---|
| **Privacy** | fully local | data sent to OpenAI | routed through OpenRouter | depends on the provider behind it |
| **Cost** | Free | Pay-per-token | Free tier + pay-per-token | provider's pricing |
| **Model variety** | Open-weight models | GPT family | 100+ providers | any supported provider |
| **Ease of setup** | Install Ollama + pull model | API key only | API key only | `pip install litellm` + provider keys |
| **Streaming** | Yes | Yes | Yes | Yes |
| **Embeddings** | with `ollama.embeddings()` | text-embedding-3 | via OpenAI/HF | `litellm.embedding()` |

**Recommended defaults:**
- **Privacy-sensitive / offline**: Ollama + Llama 3.2 / Mistral
- **Best general capability**: OpenAI GPT-4o (or Anthropic Claude through LiteLLM)
- **Free remote experimentation**: OpenRouter free-tier models
- **Model flexibility / provider-agnostic code**: LiteLLM
